# Reproducing van der Molen, Spaeth, Chini et al. (2025) from dandiset 001603

**Paper.** van der Molen T, Spaeth A, Chini M, et al. (2025). Preconfigured neuronal firing
sequences in human brain organoids. *Nature Neuroscience*.
[doi:10.1038/s41593-025-02111-0](https://doi.org/10.1038/s41593-025-02111-0). The claims file was
written from the NIH author manuscript in PubMed Central (PMC12912721.1) with the publisher's
Supplementary Information; this attempt read the same text.

**Dandiset.** [DANDI:001603](https://dandiarchive.org/dandiset/001603/0.260923.0623), version
0.260923.0623. The paper cites no version of the dandiset. This is the only published version and
the most recent one; the draft has the same 111 assets and size, so the published version is
pinned.

**What is used.** The sorted files only: each holds a 1 ms binary spike matrix
(`processing/curated_binning/t_spk_mat`, the input of the paper's code), and most also a units
table. Recordings are cut to the code's lengths (180 s for HO1 to HO4, slices and primary
cultures; 600 s for HO5 to HO8 and murine organoids). The raw `_ecephys` files are not read.

**Method sources.** The claims file (its opening comment and each criterion), and where it is
silent, the paper's Methods and Supplementary Methods and the code it links
(github.com/braingeneers/Protosequences at e9a79c4), cited in `sequences.py` and below.

**Shuffles.** The paper normalises to 100 Okun-swap shuffles per recording. This notebook uses 10
per recording, and 3 for the pairwise cross-correlations and the shuffled burst similarity, to fit
the runtime limit (a deviation; the shuffle mean is the reference, and its noise is small next to
the effects tested).

| Claim | Tested here |
|---|---|
| methods-burst-count, fig1c-example-burst-count, ed1c-lognormal-rates, fig1d-burst-duration, fig1e-peaks-near-burst-peak | tested |
| fig2-backbone-fraction, fig2e-b2b-correlation, fig2g-variance-latency, ed3f-burst-cluster-variability | tested |
| ed4b-gabazine-more-bursts, ed4b-nbqx-cpp-abolish-bursts, ed4c-gabazine-participation, ed4d-gabazine-rank-order | tested |
| supp2a-backbone-fraction-declines, supp2c-b2b-increases-with-age | tested on the human organoids only (murine longitudinal recordings are not in the dandiset) |
| supp8-backbone-stable-4h | blocked: units are not identified across the 0 h and 4 h recordings |
| ed5-shuffle-reduces-b2b, fig3e-pairwise-correlation, fig3f-backbone-pairs-in-tail | tested |
| fig4c-similarity-variance, fig4d-burst-similarity, fig4e-similarity-rises-at-backbone-onset, supp13-top-correlated-units-similarity, fig4f-pca-variance | tested |
| fig6f-slice-b2b-correlation, fig6g-slice-pairwise-correlation | tested |
| fig7a to fig7e (ten claims), supp24c-nonrigid-more-variable, supp10b-rank-order-by-model | tested |
| ed2d-alam-el-din-rates, ed2e-alam-el-din-b2b, ed2f-backbone-pairs-above-zero | tested |
| HMM claims (supp18, fig5c, fig5d, fig5f, fig7f, supp24d, supp26) | in `hmm_van_der_molen_2025.ipynb` |
| supp27-d2-near-critical | blocked (see the attempt record) |
| supp5-murine-organoid-forebrain-identity | blocked: data elsewhere (GEO), markers not listed |

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from scipy import stats

import sequences as sq

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
started = time.time()

assets = sq.list_assets()
sorted_assets = [a for a in assets if "_ecephys" not in a["path"]]
with ThreadPoolExecutor(8) as ex:
    infos = list(ex.map(sq.sorted_file_info, sorted_assets))
files = sq.recording_table(assets, infos)
print(f"dandiset {sq.DANDISET} version {sq.VERSION}: {len(assets)} assets, "
      f"{len(sorted_assets)} sorted files, {len(assets) - len(sorted_assets)} raw files")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
print(files[["path", "recording", "subject", "age", "species", "frames", "n_units", "has_units_table"]]
      .to_string(index=False))
dups = files.groupby("recording").path.apply(list)
print("\nRecordings deposited more than once:")
print(dups[dups.str.len() > 1].to_string())


def matrix_hash(asset_id):
    import hashlib
    m = sq.load_matrix(asset_id)
    return hashlib.sha1(np.packbits(m).tobytes() + str(m.shape).encode()).hexdigest()[:12]


with ThreadPoolExecutor(8) as ex:
    files["hash"] = list(ex.map(matrix_hash, files.asset_id))
same = files.groupby("hash").recording.apply(lambda s: sorted(set(s)))
print("\nGroups of files with identical spike matrices (more than one file):")
for h, recs in same.items():
    n = (files.hash == h).sum()
    if n > 1:
        print(f"  {n} files: {recs}")

Each of the HO2 and HO3 recordings at 6, 7 and 8 months and 8 months plus 4 h is deposited twice,
and the 7-month recordings a third time as `HO2_sorted.mat` and `HO3_sorted.mat`; the hashes above
show which copies hold identical spike matrices. Each recording is counted once (the first file
by path). The murine organoid MO12 has gabazine and NBQX with R-CPP sorted files but no baseline
sorted file, and no murine organoid longitudinal recording is in the dandiset.

In [ ]:
first = files.sort_values("path").groupby("recording").first()


def check_units(rec):
    row = first.loc[rec]
    m = sq.load_matrix(row.asset_id)
    st = sq.load_units_table(row.asset_id)
    out = dict(recording=rec, units=m.shape[0], seconds=m.shape[1] / 1000, spikes_matrix=int(m.sum()))
    if st is not None:
        n_tab = np.array([len(x) for x in st])
        n_mat = m.sum(1)
        out.update(spikes_table=int(n_tab.sum()),
                   units_table_max_s=max((x.max() for x in st if len(x)), default=np.nan),
                   units_matrix_le_table=bool((n_mat <= n_tab).all()),
                   lost_to_1ms_bins=int((n_tab - n_mat).sum()))
    return out


with ThreadPoolExecutor(8) as ex:
    chk = pd.DataFrame(list(ex.map(check_units, first.index)))
print(chk.to_string(index=False))
print("\nEvery matrix count is at most the units-table count (several spikes in one 1 ms bin count once):",
      bool(chk.units_matrix_le_table.dropna().all()))

The spike matrices are 1 ms bins (the file's description says so); where a units table exists,
each unit's matrix count is at most its spike count in the table, the difference being spikes
that fall in the same millisecond, and the table's last spike lies within the matrix length.
Some units tables hold a single NaN for units with no spikes. The species field says *Rattus
norvegicus* for Pr1 to Pr4 (Yuan et al., E18 Wistar rat) and *Mus musculus* for Pr5 to Pr8; the
paper calls all primary cultures murine. All files share one clock per recording: there is no
trial table, and every measure here is computed from the spike matrix alone, so a check of
alignment between streams does not apply.

In [ ]:
RECS = sq.MAIN


def run(args):
    name, rec, n_shuf, n_pair, sim, seed, light = args
    m = sq.load_matrix(first.loc[rec].asset_id, sq.keep_seconds(name))
    t0 = time.time()
    r = sq.analyse(name, m, seed, n_shuf=n_shuf, n_shuf_pair=n_pair, similarity=sim, light=light)
    r["secs"] = time.time() - t0
    return rec, r


jobs = [(n, n, 10, 3, n.startswith("HO"), 1000 + i, False) for i, n in enumerate(RECS)]
jobs += [(n, f"{n}_{c}", 10, 0, False, 2000 + 10 * i + j, True)
         for i, (n, cs) in enumerate(sq.DRUG.items()) for j, c in enumerate(cs) if f"{n}_{c}" in first.index]
jobs += [(n, f"{n}_{c}", 0, 0, False, 3000, False) for n, cs in sq.LONGITUDINAL.items() for c in cs]
res = {}
with ProcessPoolExecutor(2) as ex:
    for rec, r in ex.map(run, jobs):
        res[rec] = r
R = {k: res[k] for k in RECS}
D = {k: v for k, v in res.items() if k.split("_")[0] in sq.DRUG and "_" in k}
L = {k: v for k, v in res.items() if k.split("_")[0] in sq.LONGITUDINAL and "_" in k}
print(f"analysed {len(res)} recordings in {time.time() - started:.0f} s")

In [ ]:
summ = pd.DataFrame([dict(recording=k, type=r["mtype"], units=r["U"], seconds=r["T"] / 1000,
                          units_ge30_spikes=int((r["n_spk"] >= 30).sum()),
                          mean_rate_hz=round(r["rate"].mean(), 3), max_rate_hz=round(r["rate"].max(), 2),
                          bursts=r["n_bursts"], backbone=int(r["bb"].sum()) if "bb" in r else np.nan,
                          shuffles=len(r.get("shuf", {}).get("b2b30", r.get("shuf", {}).get("rank", []))))
                     for k, r in res.items()])
print(summ.to_string(index=False))
print("\nRecordings per model type (main set):", pd.Series([R[k]["mtype"] for k in RECS]).value_counts().to_dict())
print("Paper: 8 human organoids, 9 murine organoids, 6 slices, 8 primary cultures.")
print("Burst parameters:", sq.BURST_PARAMS, "minimum peak distance", sq.MIN_PEAK_DIST_MS, "ms")

The dandiset holds 8 human organoids, 9 murine organoids, 6 slices and 8 primary cultures, as the
paper says. Following the stated burst detection (population rate above 4 times its RMS), four
murine organoids (MO2, MO4, MO5, MO6) have no burst and MO3 has four: their population rate is
high between bursts (median unit rates above 10 Hz), so the RMS of the rate is close to its
peaks. The paper's code processed every murine organoid, so it must have used other burst
parameters for them, which the paper does not give. Claims that rest on murine organoid bursts
use the recordings that have bursts, and say so.

## methods-burst-count

> The analysed segment of each recording holds about 40 population bursts (38 ± 4, mean ± SE).

In [ ]:
nb = pd.Series({k: R[k]["n_bursts"] for k in RECS})
nb800 = pd.Series({k: R[k]["n_bursts_800"] for k in RECS})
print(pd.DataFrame({"bursts_700ms": nb, "bursts_800ms": nb800, "seconds": [R[k]["T"] / 1000 for k in RECS]}).T.to_string())
print(f"\nOver the 31 recordings: mean {nb.mean():.1f}, SE {nb.std(ddof=1) / np.sqrt(len(nb)):.1f}, "
      f"SD {nb.std(ddof=1):.1f}, range {nb.min()} to {nb.max()}")
print(f"With 800 ms minimum distance: mean {nb800.mean():.1f}, SE {nb800.std(ddof=1) / np.sqrt(len(nb800)):.1f}")
ok = nb > 0
print(f"Leaving out the four with no burst: mean {nb[ok].mean():.1f}, SE {nb[ok].std(ddof=1) / np.sqrt(ok.sum()):.1f}")
print("Expected 38 +- 4")

**Result: reproduced, numerically.** The mean is 35.7 bursts over the 31 recordings (within 38 ±
4), but with an SE of 6.2 and a range from 0 to 156: four murine organoids have no burst and two
have more than 100, so the mean matches while the recordings do not each hold about 40 bursts.

## fig1c-example-burst-count

> In the example human organoid of Figs. 1 and 2, 46 bursts occur in a three-minute interval.

In [ ]:
print(pd.DataFrame({k: dict(min_dist_700=R[k]["n_bursts"], min_dist_800=R[k]["n_bursts_800"])
                    for k in ["HO1", "HO2", "HO3", "HO4"]}).to_string())
print("Expected 46 +- 4.6")

**Result: reproduced.** HO1 has 46 bursts in its first 180 s under both peak distances; HO1 is
the example organoid for the claims that depend on this one.

In [ ]:
EXAMPLE = "HO1"

## ed1c-lognormal-rates

> In each of the 8 human organoids the distribution of unit mean firing rates is well fit by a
> lognormal distribution (R² 0.97 ± 0.04), while the rates of backbone units alone are not (R²
> 0.45 ± 0.30).

In [ ]:
HO8 = [f"HO{i}" for i in range(1, 9)]
rows = []
for units in ["all", "ge30"]:
    for reading in ["a_lognormal", "a_code_variant", "b_legend"]:
        fn = {"a_lognormal": lambda x: sq.lognormal_r2(x),
              "a_code_variant": lambda x: sq.lognormal_r2(x, variant=True),
              "b_legend": sq.lognormal_r2_legend}[reading]
        for k in HO8:
            r = R[k]
            m = np.ones(r["U"], bool) if units == "all" else r["n_spk"] >= sq.MIN_SPIKES
            rate, bb = r["rate"][m], r["bb"][m]
            rows.append(dict(units=units, reading=reading, rec=k, n=len(rate), n_bb=int(bb.sum()),
                             r2_all=fn(rate), r2_bb=fn(rate[bb]) if bb.sum() >= 3 else np.nan))
e1 = pd.DataFrame(rows)
print(e1.pivot_table(index=["units", "reading"], columns="rec", values="r2_all").round(3).to_string())
print(e1.pivot_table(index=["units", "reading"], columns="rec", values="r2_bb").round(3).to_string())
print("\nMean (SD) over organoids; backbone fits need at least 3 backbone units (HO7 has none):")
print(e1.groupby(["units", "reading"]).agg(all_mean=("r2_all", "mean"), all_sd=("r2_all", "std"),
                                           bb_mean=("r2_bb", "mean"), bb_sd=("r2_bb", "std"),
                                           n_bb_fits=("r2_bb", "count")).round(3).to_string())
print("Expected: all 0.97 +- 0.04; backbone 0.45 +- 0.30")

**Result: reproduced, under reading (a) with all units.** The mean R² is 0.949 for all units and
0.487 for backbone units (seven organoids), both within tolerance; the code's form of the density
gives 0.944 and 0.442. With units of at least 30 spikes, all units give 0.919 (outside 0.97 ±
0.04). Reading (b), the legend's normal fit to log rates, gives means of 0.60 and 0.36.

## fig1d-burst-duration

> Burst durations in the human organoids are on the order of 10² ms.

In [ ]:
dur = pd.DataFrame({k: dict(median_ms=np.median(R[k]["duration"]), min_ms=R[k]["duration"].min(),
                            max_ms=R[k]["duration"].max(), bursts=R[k]["n_bursts"])
                    for k in RECS if R[k]["n_bursts"] >= 2}).T
print(dur.to_string())
in_decade = dur.loc[["HO1", "HO2", "HO3", "HO4"], "median_ms"].between(100, 1000)
print("\nHO1 to HO4 median in 100 to 1000 ms:", in_decade.to_dict())

**Result: reproduced.** The median burst durations of HO1 to HO4 are 511, 628, 472 and 892 ms,
all within 100 to 1000 ms. Over all recordings with bursts the medians run from 104 ms (Pr4) to
2681 ms (MO3); HO5, HO6, MO3 and MO9 have medians above 1000 ms.

## fig1e-peaks-near-burst-peak

> Most units reach their peak firing rate close to the peak of the population burst.

In [ ]:
rows = []
for k in ["HO1", "HO2", "HO3", "HO4"]:
    P = R[k]["P"]
    med = np.nanmedian(P, 1)
    med = med[np.isfinite(med)]
    pooled = P[np.isfinite(P)]
    for bound in [50, 100]:
        rows.append(dict(rec=k, bound_ms=bound, units=len(med), median_within=np.mean(np.abs(med) <= bound),
                         unit_bursts=len(pooled), pooled_within=np.mean(np.abs(pooled) <= bound)))
f1e = pd.DataFrame(rows)
print(f1e.round(3).to_string(index=False))
print("\nReading and organoid combinations above one half:",
      int((f1e.median_within > 0.5).sum() + (f1e.pooled_within > 0.5).sum()), "of", 2 * len(f1e))

**Result: partially reproduced.** In 15 of the 16 combinations of organoid, quantity and bound,
more than half of the values lie within the bound (0.53 to 0.89); the exception is HO4 with
pooled unit-burst peak times within 50 ms (0.481). The criterion asks for every reading.

## fig2-backbone-fraction

> Backbone units, which fire at least twice in every burst, are 28% ± 14% (mean ± STD) of the
> units in the 8 human organoids.

In [ ]:
bf = pd.DataFrame({k: dict(units=R[k]["U"], units_ge30=int((R[k]["n_spk"] >= 30).sum()),
                           backbone=int(R[k]["bb"].sum()), frac_all=R[k]["bb"].mean(),
                           frac_ge30=R[k]["bb"][R[k]["n_spk"] >= 30].mean()) for k in HO8}).T
print(bf.to_string())
for c in ["frac_all", "frac_ge30"]:
    print(f"{c}: mean {bf[c].mean():.3f}, SD {bf[c].astype(float).std(ddof=1):.3f} (population SD {bf[c].astype(float).std(ddof=0):.3f})")
print("Expected mean 0.28 +- 0.14, SD 0.14 +- 0.014")

**Result: reproduced, under reading (b).** With units of at least 30 spikes as the denominator
the mean is 0.331 and the SD 0.150, both within tolerance. With all sorted units the mean is
0.258 (within) and the SD 0.166 (outside 0.126 to 0.154). HO7 has no backbone unit here (no unit
fires twice in all 24 bursts).

## fig2e-b2b-correlation

> Backbone units have higher burst-to-burst correlations of their firing rate than non-rigid units.

In [ ]:
def unit_table(keys, bbkey="bb"):
    rows = []
    for k in keys:
        r = R[k]
        if "bb" not in r:
            continue
        sh = r.get("shuf", {})
        b2b_sh = np.nanmean(np.array(sh["b2b30"]), 0) if sh.get("b2b30") else np.full(r["U"], np.nan)
        cv_sh = np.array(sh["cv2"]) if sh.get("cv2") else None
        for u in range(r["U"]):
            rows.append(dict(rec=k, mtype=r["mtype"], unit=f"{k}_{u}", n_spk=r["n_spk"][u], rate=r["rate"][u],
                             bb=int(r[bbkey][u]), b2b30=r["b2b30"][u], b2b70=r["b2b70"][u], b2b_shuf=b2b_sh[u],
                             cv2=r["cv2"][u],
                             cv2_shuf_mean=np.nanmean(cv_sh[:, u]) if cv_sh is not None else np.nan,
                             cv2_shuf_sd=np.nanstd(cv_sh[:, u], ddof=1) if cv_sh is not None else np.nan))
    return pd.DataFrame(rows)


HO4 = ["HO1", "HO2", "HO3", "HO4"]
ut = unit_table(HO4)
for rule in ["b2b30", "b2b70"]:
    d = ut[np.isfinite(ut[rule])].rename(columns={rule: "y"})
    f = sq.lmm(d, "y ~ bb", "rec")
    print(f"{rule}: {len(d)} units ({d.bb.sum()} backbone); mean backbone {d[d.bb == 1].y.mean():.3f}, "
          f"non-rigid {d[d.bb == 0].y.mean():.3f}; LMM effect {f.params['bb']:.3f}, z {f.tvalues['bb']:.1f}, p {f.pvalues['bb']:.2g}")
    print("   per organoid:", d.groupby(["rec", "bb"]).y.mean().round(3).unstack().to_dict("index"))
print(f"\nAll sorted units in HO1 to HO4: {sum(R[k]['U'] for k in HO4)} (expected 507 +- 51)")

**Result: reproduced.** Backbone units have higher burst-to-burst correlations than non-rigid
units (0.796 against 0.574, p = 6e-36, 216 units under the Methods' 30% rule; 0.796 against
0.629, p = 2e-19, 150 units under the code's 70% rule), in each organoid. The paper's n of 507
is the number of all sorted units in HO1 to HO4 (507 here as well); the units that pass the
rules are fewer.

## fig2g-variance-latency

> Among backbone units, the variance of firing-rate peak times across bursts grows with the
> unit's position in the sequence.

In [ ]:
for reading in ["a_rank", "b_rescaled"]:
    rows = []
    for k in HO4:
        r = R[k]
        P = r["P"][r["bb"]]
        med = np.nanmedian(P, 1)
        var = np.nanvar(P, 1, ddof=1)
        if reading == "a_rank":
            pos = (stats.rankdata(med) - 1) / (len(med) - 1)
        else:
            pos = (med - med.min()) / (med.max() - med.min())
        rows += [dict(rec=k, pos=a, var=v) for a, v in zip(pos, var)]
    d = pd.DataFrame(rows)
    f = sq.lmm(d, "var ~ pos", "rec")
    fl = sq.lmm(d.assign(lv=np.log10(d["var"] + 1)), "lv ~ pos", "rec")
    print(f"{reading}: {len(d)} backbone units; slope {f.params['pos']:.0f} ms^2 per unit position, p {f.pvalues['pos']:.2g}; "
          f"log10 variance slope {fl.params['pos']:.2f}, p {fl.pvalues['pos']:.2g}")
    print("   Spearman per organoid:", {k: round(stats.spearmanr(g.pos, g["var"])[0], 2) for k, g in d.groupby("rec")})

**Result: reproduced.** The variance of backbone peak times rises with relative position under
both readings (rank, p = 4e-4; rescaled median time, p = 5e-5; on log variance p below 1e-6),
with positive rank correlations in each organoid.

## ed3f-burst-cluster-variability

> When bursts are clustered by their pairwise unit correlation matrices, the firing rates of
> non-rigid units vary more between clusters than those of backbone units.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


def burst_corr_vectors(seg):
    """Per burst, the lower triangle of the unit-by-unit correlation of the -250 to 500 ms rates
    (numpy corrcoef; undefined correlations set to 0, as the code's nanmax with the identity)."""
    U = seg.shape[0]
    tri = np.tril_indices(U, -1)
    out = []
    for b in range(seg.shape[1]):
        with np.errstate(invalid="ignore", divide="ignore"):
            c = np.corrcoef(seg[:, b, :])
        c = np.nan_to_num(c, nan=0.0)
        out.append(c[tri])
    return np.array(out)


def kneedle(ks, sse):
    """Knee of a decreasing convex curve: the point farthest below the chord (Satopaa et al.)."""
    x = (np.array(ks) - ks[0]) / (ks[-1] - ks[0])
    y = (np.array(sse) - min(sse)) / (max(sse) - min(sse))
    return ks[int(np.argmax((1 - x) - y))]


rows, kinfo = [], []
for k in HO8:
    r = R[k]
    seg = r["segments"]
    V = burst_corr_vectors(seg)
    ks = list(range(2, min(10, len(V) - 1) + 1))
    sse, sil, labels = [], [], {}
    for kk in ks:
        best = None
        for seed in range(10):
            km = KMeans(n_clusters=kk, init="k-means++", random_state=seed, n_init=1).fit(V)
            if best is None or km.inertia_ < best.inertia_:
                best = km
        sse.append(best.inertia_)
        sil.append(silhouette_score(V, best.labels_))
        labels[kk] = best.labels_
    choice = {"a_kneedle": kneedle(ks, sse), "b_silhouette": ks[int(np.argmax(sil))]}
    kinfo.append(dict(rec=k, bursts=len(V), **choice))
    mean_rate = seg.mean(2)  # units by bursts
    for reading, kk in choice.items():
        lab = labels[kk]
        cm = np.array([mean_rate[:, lab == c].mean(1) for c in range(kk)]).T
        with np.errstate(invalid="ignore", divide="ignore"):
            cv = cm.std(1) / cm.mean(1)
        rows += [dict(reading=reading, rec=k, bb=int(r["bb"][u]), cv=cv[u]) for u in range(r["U"]) if np.isfinite(cv[u])]
print(pd.DataFrame(kinfo).to_string(index=False))
cvd = pd.DataFrame(rows)
for reading in ["a_kneedle", "b_silhouette"]:
    for recs, lab in [(HO8, "HO1-HO8"), (HO4, "HO1-HO4")]:
        d = cvd[(cvd.reading == reading) & cvd.rec.isin(recs)]
        f = sq.lmm(d, "cv ~ bb", "rec")
        print(f"{reading} {lab}: {len(d)} units; CV backbone {d[d.bb == 1].cv.mean():.3f}, non-rigid {d[d.bb == 0].cv.mean():.3f}; "
              f"effect of backbone {f.params['bb']:.3f}, p {f.pvalues['bb']:.2g}")

**Result: reproduced.** Non-rigid units vary more between burst clusters than backbone units
under both elbow readings and both organoid sets (mean CV 0.70 to 1.31 against 0.07 to 0.31, p
from 5e-48 to 4e-94). The elbow picked by the kneedle rule (4 to 7 clusters) and by the largest
silhouette (2 to 10) differ, and the outcome is the same.

## Murine organoids with gabazine and with NBQX and R-CPP (ed4b, ed4c, ed4d)

In [ ]:
drug = pd.DataFrame([dict(rec=k, organoid=k.split("_")[0], cond=k.split("_")[1], units=r["U"],
                          seconds=r["T"] / 1000, bursts=r["n_bursts"],
                          bursts_per_min=r["n_bursts"] / (r["T"] / 60000)) for k, r in D.items()])
print(drug.to_string(index=False))
print("\nMO12 has no baseline sorted file. Unit counts are equal across the conditions of each organoid.")

## ed4b-gabazine-more-bursts

> In murine organoids, blocking GABA-A receptors with gabazine increases the number of population
> bursts.

In [ ]:
d = drug[drug.cond.isin(["BASE", "GABAZINE"])].copy()
d["gab"] = (d.cond == "GABAZINE").astype(int)
print(d.pivot(index="organoid", columns="cond", values="bursts_per_min").round(2).to_string())
f = sq.lmm(d, "bursts_per_min ~ gab", "organoid")
print(f"\nBurst rate, gabazine minus baseline: {f.params['gab']:.2f} per min, p {f.pvalues['gab']:.2g} "
      f"({d.organoid.nunique()} organoids, {len(d)} recordings)")
pair = d.pivot(index="organoid", columns="cond", values="bursts_per_min").dropna()
print("Paired t-test over the organoids with both:", stats.ttest_rel(pair.GABAZINE, pair.BASE))
fc = sq.lmm(d, "bursts ~ gab", "organoid")
print(f"Burst count: {fc.params['gab']:.1f}, p {fc.pvalues['gab']:.2g}")

**Result: reproduced.** Gabazine raises the burst rate in each of the four organoids with a
baseline recording (by 2.5 to 10.4 bursts per minute; 7.0 per minute in the mixed model, p =
6e-6; paired t-test over the four, p = 0.023).

## ed4b-nbqx-cpp-abolish-bursts

> Blocking AMPA and NMDA receptors with NBQX and R-CPP abolishes population bursts in murine
> organoids.

In [ ]:
d = drug[drug.cond.isin(["BASE", "NBQXCPP"])].copy()
d["nbqx"] = (d.cond == "NBQXCPP").astype(int)
print(d.pivot(index="organoid", columns="cond", values="bursts").to_string())
f = sq.lmm(d, "bursts_per_min ~ nbqx", "organoid")
print(f"\nBurst rate, NBQX with R-CPP minus baseline: {f.params['nbqx']:.2f} per min, p {f.pvalues['nbqx']:.2g}")
print("Treated recordings with at most one burst:", int((drug[drug.cond == "NBQXCPP"].bursts <= 1).sum()),
      "of", int((drug.cond == "NBQXCPP").sum()))

**Result: reproduced.** Every NBQX with R-CPP recording has no burst or one (MO12), against 81 to
116 bursts at baseline, and the fall is significant.

## ed4c-gabazine-participation

> Gabazine increases the fraction of bursts in which units fire at least two spikes.

In [ ]:
rows = []
for k, r in D.items():
    if "frac2" not in r or k.endswith("NBQXCPP"):
        continue
    o, c = k.split("_")
    rows += [dict(organoid=o, cond=c, unit=f"{o}_{u}", frac=r["frac2"][u], gab=int(c == "GABAZINE"))
             for u in range(r["U"])]
pd4 = pd.DataFrame(rows)
print(pd4.groupby(["organoid", "cond"]).frac.agg(["count", "mean"]).round(3).unstack().to_string())
f = sq.lmm(pd4, "frac ~ gab", "organoid")
print(f"\nPooled per condition (main reading): gabazine minus baseline {f.params['gab']:.3f}, p {f.pvalues['gab']:.2g}")
m = pd4[pd4.organoid.isin(["MO10", "MO11", "MO14"])]
fm = sq.lmm(m, "frac ~ gab", "organoid")
fu = sq.lmm(m, "frac ~ gab", "unit")
print(f"Variant, MO10, MO11 and MO14 (unit positions identical across conditions), random intercept per unit: "
      f"{fu.params['gab']:.3f}, p {fu.pvalues['gab']:.2g}")

**Result: reproduced.** With units pooled per condition, the fraction of bursts in which a unit
fires at least twice rises with gabazine by 0.138 (p = 9e-4), in each of the four organoids with
a baseline. Treating units as matched in the three organoids whose unit positions are identical
across conditions (which the dataset does not state) gives 0.150, p = 1e-8.

## ed4d-gabazine-rank-order

> Gabazine makes the order of backbone sequences more consistent across bursts.

In [ ]:
def rank_table(src, keys):
    rows = []
    for k in keys:
        r = src[k]
        if r.get("rank") is None or not r["shuf"]["rank"]:
            continue
        A = r["rank"]
        S = np.array(r["shuf"]["rank"])
        B = np.nanmean(S, 0)
        sd = np.nanstd(S, 0, ddof=1)
        with np.errstate(invalid="ignore", divide="ignore"):
            z = (A - B) / sd
        rows.append(pd.DataFrame(dict(rec=k, raw=A, shuf=B, norm=sq.normalize(A, B), z=z)))
    return pd.concat(rows)


rk = rank_table(D, [k for k in D if not k.endswith("NBQXCPP")])
rk["organoid"] = rk.rec.str.split("_").str[0]
rk["gab"] = rk.rec.str.endswith("GABAZINE").astype(int)
print(rk.groupby("rec")[["raw", "shuf", "norm", "z"]].median().round(3).to_string())
for v in ["norm", "z"]:
    d = rk[np.isfinite(rk[v])]
    f = sq.lmm(d, f"{v} ~ gab", "organoid")
    print(f"{v}: {len(d)} burst pairs; gabazine minus baseline {f.params['gab']:.3f}, p {f.pvalues['gab']:.2g}")

**Result: blocked (methods underspecified).** The raw rank-order correlation of backbone
sequences rises with gabazine in every organoid (medians 0.07 to 0.35 at baseline, 0.55 to 0.91
with gabazine). Normalised by z-score to the shuffles, gabazine is higher (by 2.0, p below 1e-300).
Normalised as (A - B)/(A + B), it is not (-1.14, p = 0.6): the shuffled correlations average near
0, so A + B is near 0 for many burst pairs and the ratio takes extreme values. The criterion asks
for both readings and they disagree.

## supp2a-backbone-fraction-declines and supp2c-b2b-increases-with-age (human organoids)

In [ ]:
lg = pd.DataFrame([dict(rec=k, organoid=k.split("_")[0], age=k.split("_", 1)[1], units=r["U"], bursts=r["n_bursts"],
                        backbone=int(r["bb"].sum()), frac_all=r["bb"].mean(),
                        frac_ge30=r["bb"][r["n_spk"] >= 30].mean()) for k, r in L.items()])
print(lg.to_string(index=False))
dev = lg[lg.age.isin(["6M", "7M", "8M"])]
for c in ["frac_all", "frac_ge30"]:
    t = dev.pivot(index="organoid", columns="age", values=c)
    print(f"\n{c}:\n{t.round(3).to_string()}")
    print("  (a) mean lower at 8M than 6M:", bool(t["8M"].mean() < t["6M"].mean()),
          "; (b) lower in every organoid:", bool((t["8M"] < t["6M"]).all()))
print("\nMurine organoid longitudinal recordings: none in the dandiset (MO1 to MO14 are all age P45D).")

**Result: partially reproduced.** In the human organoids HO2 and HO3 the backbone fraction is
lower at 8 months than at 6 months, on average and in each organoid, with either denominator.
The murine part cannot be tested: the dandiset holds no longitudinal murine organoid recordings.

In [ ]:
from statsmodels.formula.api import ols
import statsmodels.api as sm

rows = []
for k, r in L.items():
    o, age = k.split("_", 1)
    if age not in ("6M", "7M", "8M"):
        continue
    rows += [dict(organoid=o, age=age, months=int(age[0]), bb=int(r["bb"][u]), y=r["b2b30"][u])
             for u in range(r["U"]) if np.isfinite(r["b2b30"][u])]
ag = pd.DataFrame(rows)
print(ag.groupby(["organoid", "age", "bb"]).y.agg(["count", "mean"]).round(3).unstack().to_string())
print(ag.groupby(["age", "bb"]).y.mean().round(3).unstack().to_string())
for second in ["C(bb)", "C(organoid)"]:
    fit = ols(f"y ~ C(age) + {second}", ag).fit()
    print(f"\nTwo-way ANOVA, age and {second}:\n", sm.stats.anova_lm(fit, typ=2).to_string(float_format=lambda v: f"{v:.3g}"))
mono = ag.groupby("age").y.mean()
print("\nMean by age:", mono.round(3).to_dict())
print("Backbone above non-rigid at every age:", bool((ag.groupby(["age", "bb"]).y.mean().unstack().diff(axis=1)[1] > 0).all()))

**Result: partially reproduced.** In HO2 and HO3 the burst-to-burst correlation rises with age
(means 0.61, 0.69 and 0.72 at 6, 7 and 8 months; age effect F = 18.3, p = 3e-8 with unit type as
the second factor, F = 12.8, p = 5e-6 with organoid), and backbone units are above non-rigid
units at every age. The murine part cannot be tested.

## ed5-shuffle-reduces-b2b

> Randomizing spike times while keeping unit and population rates reduces burst-to-burst
> correlations.

In [ ]:
u8 = unit_table(HO8)
d = u8[(u8.bb == 1) & np.isfinite(u8.b2b30) & np.isfinite(u8.b2b_shuf)]
long = pd.concat([d.assign(y=d.b2b30, real=1), d.assign(y=d.b2b_shuf, real=0)])
fa = sq.lmm(long, "y ~ real", "rec")
fa_unit = sq.lmm(long, "y ~ real", "unit")
print(f"Backbone units with both values: {len(d)}; real {d.b2b30.mean():.3f}, shuffled mean {d.b2b_shuf.mean():.3f}")
print(f"(a) real minus shuffled, random intercept per organoid: {fa.params['real']:.3f}, p {fa.pvalues['real']:.2g}; "
      f"per unit: {fa_unit.params['real']:.3f}, p {fa_unit.pvalues['real']:.2g}")
d = d.assign(norm=sq.normalize(d.b2b30, d.b2b_shuf))
fb = sq.lmm(d, "norm ~ 1", "rec")
print(f"(b) normalised mean {fb.params['Intercept']:.3f}, one-sided p {fb.pvalues['Intercept'] / 2 if fb.params['Intercept'] > 0 else 1 - fb.pvalues['Intercept'] / 2:.2g}")
print("Per organoid, real minus shuffled (backbone):", d.groupby("rec").apply(lambda g: round((g.b2b30 - g.b2b_shuf).mean(), 3)).to_dict())
dn = u8[(u8.bb == 0) & np.isfinite(u8.b2b30) & np.isfinite(u8.b2b_shuf)]
print(f"Non-rigid units: real {dn.b2b30.mean():.3f}, shuffled {dn.b2b_shuf.mean():.3f}")

**Result: reproduced.** For backbone units of HO1 to HO8 the real burst-to-burst correlation
(0.617) exceeds the mean over shuffles (0.513): by 0.104 with a random intercept per organoid (p =
9e-26) or per unit (p = 2e-28), and the normalised value is above 0 (0.076, one-sided p = 0.007).
HO8 goes the other way (-0.06); HO7 has no backbone unit.

## fig3e-pairwise-correlation

> Pairwise cross-correlations of firing rates are higher between backbone units than between
> backbone and non-rigid units or between non-rigid units.

In [ ]:
def pair_table(keys, bbkey="bb", src=None):
    src = R if src is None else src
    out = []
    for k in keys:
        r = src[k]
        if "pw_corr" not in r:
            continue
        idx = r["idx30"]
        bb = r[bbkey][idx]
        iu = np.triu_indices(len(idx), 1)
        t = np.where(bb[iu[0]] & bb[iu[1]], "bb", np.where(bb[iu[0]] | bb[iu[1]], "mix", "nr"))
        sh = r.get("shuf", {})
        Bc = np.nanmean(np.array(sh["pw_corr"]), 0)[iu] if sh.get("pw_corr") else np.nan
        Bl = np.nanmean(np.array(sh["pw_lag"]), 0)[iu] if sh.get("pw_lag") else np.nan
        out.append(pd.DataFrame(dict(rec=k, mtype=r["mtype"], type=t, y=r["pw_corr"][iu],
                                     lag=np.abs(r["pw_lag"][iu]), shuf=Bc, shuf_lag=Bl)))
    return pd.concat(out, ignore_index=True)


def type_contrasts(d, group="rec", y="y"):
    f = sq.lmm(d.rename(columns={y: "yy"}), "yy ~ 0 + C(type)", group)
    out = {}
    for other in ["mix", "nr"]:
        if f"C(type)[{other}]" in f.fe_params.index:
            c = sq.contrast(f, {"C(type)[bb]": 1.0, f"C(type)[{other}]": -1.0})
            out[f"bb-{other}"] = (round(c["estimate"], 3), float(f"{c['p']:.2g}"))
    return out


pt = pair_table(HO4)
d = pt[np.isfinite(pt.y)]
print(f"{len(d)} pairs among units with at least 30 spikes; mean by type:", d.groupby("type").y.mean().round(3).to_dict())
print("Contrasts (estimate, p):", type_contrasts(d))
print("Per organoid:", d.groupby(["rec", "type"]).y.mean().round(3).unstack().to_dict("index"))
n_all = sum(R[k]["U"] * (R[k]["U"] - 1) // 2 for k in HO4)
print(f"All unit pairs of HO1 to HO4 (every sorted unit): {n_all} (expected 34056 +- 3406)")

**Result: reproduced.** Backbone pairs (0.70) exceed mixed (0.36) and non-rigid pairs (0.23),
both contrasts with p below 1e-20, in each organoid. The paper's 34,056 pairs is the number of
pairs among all sorted units of HO1 to HO4 (34,056 here as well); 9,648 pairs among units with
at least 30 spikes enter the test.

## fig3f-backbone-pairs-in-tail

> The distribution of pairwise correlations is skewed and roughly lognormal, and backbone-backbone
> pairs lie in its upper tail in every organoid.

In [ ]:
rows = []
for k in HO4:
    x = d[d.rec == k]
    m = x[x.type == "bb"].y.mean()
    lx = np.log(x.y[x.y > 0])
    h, e = np.histogram(lx, bins=30)
    cen = (e[:-1] + e[1:]) / 2
    yh = stats.norm.pdf(cen, lx.mean(), lx.std()) * len(lx) * np.diff(e)[0]
    rows.append(dict(rec=k, pairs=len(x), bb_mean=m, p75=np.percentile(x.y, 75), p90=np.percentile(x.y, 90),
                     percentile_of_bb_mean=stats.percentileofscore(x.y, m), skew=stats.skew(x.y),
                     r2_normal_on_log=1 - ((h - yh) ** 2).sum() / ((h - h.mean()) ** 2).sum()))
f3f = pd.DataFrame(rows)
print(f3f.round(3).to_string(index=False))
print("Above 75th percentile in all:", bool((f3f.bb_mean > f3f.p75).all()), "; above 90th in all:", bool((f3f.bb_mean > f3f.p90).all()))

**Result: partially reproduced.** The backbone-pair mean lies above the 75th percentile of all
pairs in every organoid (at the 87th to 95th percentile), and above the 90th percentile in three
of four (HO3 at the 87th). The distributions are right-skewed (skew 0.26 to 1.08).

## Burst similarity (fig4c, fig4d, fig4e, supp13)

In [ ]:
def bb_window_frames(r):
    lo, hi = sq.backbone_window(r["P"], r["bb"])
    return lo, hi


rows = []
for k in HO8:
    r = R[k]
    if r.get("sim_bb") is None:
        continue
    lo, hi = bb_window_frames(r)
    fr = sq.SIM_FRAMES
    w = (fr >= lo) & (fr <= hi)
    rows.append(dict(rec=k, window=f"{lo} to {hi} ms", bursts=r["n_bursts"],
                     all=np.nanmean(r["sim_all"][w]), backbone=np.nanmean(r["sim_bb"][w]),
                     nonrigid=np.nanmean(r["sim_nr"][w]), shuffled=np.nanmean(r["shuf"]["sim_all"][w])))
print("Mean similarity over the backbone window:")
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("HO7 has no backbone unit, so no backbone window.")

## fig4c-similarity-variance

> The spread of burst similarity across pairs of bursts is high outside bursts, drops at burst
> onset and stays low until the bursts end.

In [ ]:
rows = []
for k in HO8:
    r = R[k]
    b = r["bursts"]
    fr = sq.SIM_FRAMES
    rel_s, rel_e = b["start"] - b["peak"], b["end"] - b["peak"]
    span = (fr >= max(rel_s.min(), fr[0])) & (fr <= min(rel_e.max(), fr[-1]))
    sd = np.nanstd(r["sim_all"], 1)
    readings = {"a_median_burst": (np.median(rel_s), np.median(rel_e))}
    if r["bb"].sum():
        readings["b_backbone_window"] = bb_window_frames(r)
    for name, (lo, hi) in readings.items():
        inside = span & (fr >= lo) & (fr <= hi)
        outside = span & ~inside
        rows.append(dict(rec=k, reading=name, inside=f"{lo:.0f} to {hi:.0f}", sd_inside=sd[inside].mean(),
                         sd_outside=sd[outside].mean() if outside.any() else np.nan, frames_outside=int(outside.sum())))
f4c = pd.DataFrame(rows)
print(f4c.round(3).to_string(index=False))
ex = f4c[f4c.rec == EXAMPLE]
print("\nExample organoid, inside lower than outside under both readings:", bool((ex.sd_inside < ex.sd_outside).all()))

**Result: reproduced.** In the example organoid HO1 the standard deviation of burst similarity
over burst pairs is lower inside the burst than outside it, under both readings (0.142 against
0.191 with the median burst edges, 0.133 against 0.163 with the backbone window); the same holds
in every organoid shown.

## fig4d-burst-similarity

> Across pairs of bursts, the population firing vectors of backbone units are more similar
> throughout the backbone window than those of non-rigid units or of shuffled data.

In [ ]:
rows = []
for k in HO8:
    r = R[k]
    if r.get("sim_bb") is None:
        continue
    lo, hi = bb_window_frames(r)
    fr = sq.SIM_FRAMES
    w = np.nonzero((fr >= lo) & (fr <= hi))[0]
    alpha = 0.05 / len(w)
    res_k = {}
    for other, X in [("nonrigid", r["sim_nr"]), ("shuffled", r["shuf"]["sim_all"])]:
        ok = 0
        for i in w:
            a, c = r["sim_bb"][i], X[i]
            m = np.isfinite(a) & np.isfinite(c)
            t, p = stats.ttest_rel(a[m], c[m])
            ok += int(t > 0 and p < alpha)
        res_k[other] = ok / len(w)
    rows.append(dict(rec=k, frames=len(w), frac_frames_bb_gt_nonrigid=res_k["nonrigid"],
                     frac_frames_bb_gt_shuffled=res_k["shuffled"]))
f4d = pd.DataFrame(rows)
print(f4d.round(3).to_string(index=False))

**Result: reproduced.** In HO1, backbone similarity is significantly higher than non-rigid and
than shuffled similarity at every one of the 226 frames of the backbone window (Bonferroni). The
same holds at every frame in HO2 to HO6; in HO8 at 67% and 21% of frames.

## fig4e-similarity-rises-at-backbone-onset

> In every human organoid the burst similarity rises from the onset of the backbone period and
> stays high during it.

In [ ]:
rows = []
for k in HO8:
    r = R[k]
    if not r["bb"].sum():
        rows.append(dict(rec=k, note="no backbone unit, no backbone period"))
        continue
    lo, hi = bb_window_frames(r)
    fr = sq.SIM_FRAMES
    m = np.nanmean(r["sim_all"], 1)
    w = (fr >= lo) & (fr <= hi)
    first_val = m[np.nonzero(fr == lo)[0][0]]
    rows.append(dict(rec=k, window=f"{lo} to {hi}", at_onset=first_val, mean_in_period=m[w].mean(),
                     rises=bool(m[w].mean() > first_val)))
print(pd.DataFrame(rows).round(3).to_string(index=False))

**Result: partially reproduced.** In the seven organoids that have backbone units, the mean
similarity over the backbone period exceeds its value at the period's first frame (in HO8 barely,
0.329 against 0.326). HO7 has no backbone unit, so no backbone period, and the claim cannot hold
in all 8.

## supp13-top-correlated-units-similarity

> The 20% of units with the highest average pairwise correlation give a higher burst similarity
> than all units together, and the 20% with the lowest give a lower one.

In [ ]:
r = R[EXAMPLE]
m_ex = sq.load_matrix(first.loc[EXAMPLE].asset_id, sq.keep_seconds(EXAMPLE))
Z = sq.zscore_rows(sq.isi_rates(m_ex))
idx = r["idx30"]
C = r["pw_corr"].copy()
C = np.where(np.isnan(C), C.T, C)
np.fill_diagonal(C, np.nan)
avg = np.nanmean(C, 1)
order = idx[np.argsort(avg)]  # ascending mean pairwise correlation
lo, hi = bb_window_frames(r)
fr = sq.SIM_FRAMES
w = np.nonzero((fr >= lo) & (fr <= hi))[0]
peaks = r["bursts"]["peak"]
sim_all = r["sim_all"][w]


def subset_sim(units):
    return sq.burst_similarity(Z[units], peaks, fr[w])


n20 = int(round(0.2 * len(order)))
top, bot = subset_sim(order[-n20:]), subset_sim(order[:n20])
alpha = 0.05 / len(w)
res13 = {}
for name, X, sign in [("top20_higher", top, 1), ("bottom20_lower", bot, -1)]:
    ok = 0
    for i in range(len(w)):
        mm = np.isfinite(X[i]) & np.isfinite(sim_all[i])
        t, p = stats.ttest_rel(X[i][mm], sim_all[i][mm])
        ok += int(sign * t > 0 and p < alpha)
    res13[name] = ok / len(w)
print(f"{EXAMPLE}: {len(order)} units with at least 30 spikes, 20% = {n20}; backbone window {lo} to {hi} ms ({len(w)} frames)")
print("Mean similarity in the window: all", round(np.nanmean(sim_all), 3), "top 20%", round(np.nanmean(top), 3),
      "bottom 20%", round(np.nanmean(bot), 3))
print("Fraction of window frames significant (Bonferroni):", {k: round(v, 3) for k, v in res13.items()})
diffs = {}
for pct in [20, 40, 60, 80, 95]:
    n = max(1, int(round(pct / 100 * len(order))))
    diffs[pct] = round(float(np.nansum(np.nanmean(subset_sim(order[-n:]), 1)) - np.nansum(np.nanmean(subset_sim(order[:n]), 1))), 2)
print("Summed similarity over the window, top minus bottom, by percentile:", diffs)
del Z, m_ex

**Result: reproduced.** In HO1 the top 20% of units by mean pairwise correlation (19 units) give
a higher burst similarity than all units (0.868 against 0.453) and the bottom 20% a lower one
(0.056), each significant at every frame of the backbone window; the top-minus-bottom difference
in summed similarity shrinks from 183.6 at 20% to 6.3 at 95%.

## fig4f-pca-variance

> In the example organoid, the first two principal components of backbone unit rates explain 73%
> of their variance against 25% for non-rigid units, and across organoids backbone units need
> fewer components than all units while non-rigid units need more.

In [ ]:
rows = []
for k in HO8:
    p = R[k]["pca"]
    rows.append(dict(rec=k, **{f"{s}_2pc": (100 * p[s][:2].sum() if p[s] is not None else np.nan) for s in ["all", "bb", "nr"]}))
f4f = pd.DataFrame(rows)
f4f["bb_gt_all"] = f4f.bb_2pc > f4f.all_2pc
f4f["nr_lt_all"] = f4f.nr_2pc < f4f.all_2pc
print(f4f.round(1).to_string(index=False))
print("Expected for the example organoid: backbone 73 +- 7.3, non-rigid 25 +- 2.5 percent")
match = f4f[(f4f.bb_2pc.between(65.7, 80.3)) & (f4f.nr_2pc.between(22.5, 27.5))]
print("Organoids matching both:", match.rec.tolist())

**Result: partially reproduced.** In HO1 the first two PCs explain 75.4% of the backbone units'
variance (within 73 ± 7.3) but 17.4% for non-rigid units (outside 25 ± 2.5); no organoid matches
both values. In each of the seven organoids with backbone units, backbone units have a higher and
non-rigid units a lower two-PC variance than all units; HO7 has no backbone unit.

## Neonatal slices (fig6f, fig6g)

In [ ]:
MS = ["M1S1", "M1S2", "M2S1", "M2S2", "M3S1", "M3S2"]
for key, lab in [("bb", "90% (Methods)"), ("bb_70", "70% (Fig. 6B legend)")]:
    us = unit_table(MS, key)
    us["animal"] = us.rec.str[:2]
    d = us[np.isfinite(us.b2b30)]
    for g in ["rec", "animal"]:
        f = sq.lmm(d.rename(columns={"b2b30": "y"}), "y ~ bb", g)
        print(f"{lab}, random intercept per {g}: {len(d)} units ({int(d.bb.sum())} backbone); backbone {d[d.bb == 1].b2b30.mean():.3f}, "
              f"non-rigid {d[d.bb == 0].b2b30.mean():.3f}; effect {f.params['bb']:.3f}, p {f.pvalues['bb']:.2g}")
print("Per slice (90%):", unit_table(MS).dropna(subset=["b2b30"]).groupby(["rec", "bb"]).b2b30.mean().round(3).unstack().to_dict("index"))
print(f"All sorted units in the six slices: {sum(R[k]['U'] for k in MS)} (expected 959 +- 96)")

**Result: reproduced.** Backbone units have higher burst-to-burst correlations under both
thresholds and both random-effect structures (p from 2e-9 to 1e-26). The paper's n of 959 is the
number of all sorted units in the six slices (959 here); 715 pass the unit rules.

In [ ]:
for key, lab in [("bb", "90%"), ("bb_70", "70%")]:
    ps = pair_table(MS, key)
    ps["animal"] = ps.rec.str[:2]
    d = ps[np.isfinite(ps.y)]
    print(f"{lab}: {len(d)} pairs; means {d.groupby('type').y.mean().round(3).to_dict()}")
    for g in ["rec", "animal"]:
        print(f"   random intercept per {g}: contrasts {type_contrasts(d, g)}")
n_all = sum(R[k]["U"] * (R[k]["U"] - 1) // 2 for k in MS)
print(f"All unit pairs in the six slices: {n_all} (expected 125607 +- 12561)")

**Result: reproduced.** Backbone pairs exceed mixed and non-rigid pairs under every reading.
The paper's 125,607 pairs is the number of pairs among all sorted units of the slices (125,607
here); the pairs among units with at least 30 spikes are fewer.

## Comparisons across model types (Fig. 7)

Recordings without bursts (MO2, MO4, MO5, MO6) have no backbone units and are left out; MO3
(four bursts) is kept. Slices are shown at both backbone thresholds.

In [ ]:
ALL = [k for k in RECS if R[k]["n_bursts"] >= 2]


def unit_all(slice_key="bb"):
    parts = []
    for k in ALL:
        key = slice_key if R[k]["mtype"] == "MS" else "bb"
        parts.append(unit_table([k], key))
    return pd.concat(parts, ignore_index=True)


def within_type(d, y, types=("HO", "MO", "MS", "PR")):
    d = d.copy()
    d["cell"] = d.mtype + "_" + np.where(d.bb == 1, "bb", "nr")
    f = sq.lmm(d.rename(columns={y: "yy"}), "yy ~ 0 + C(cell)", "rec")
    return f, {t: sq.cell_means_contrast(f, f"{t}_bb", f"{t}_nr") for t in types}


print("Recordings in the Fig. 7 comparisons:", len(ALL), ALL)

### fig7a-backbone-higher-rate and fig7a-primary-higher-rate

In [ ]:
f7a = {}
for sk in ["bb", "bb_70"]:
    ua = unit_all(sk)
    d = ua[ua.n_spk >= sq.MIN_SPIKES].assign(lr=lambda x: np.log10(x.rate))
    f, con = within_type(d, "lr")
    f7a[sk] = (d, f)
    print(f"Slices at {'90' if sk == 'bb' else '70'}%: units per type {d.groupby('mtype').size().to_dict()}")
    for t, c in con.items():
        print(f"   {t}: backbone minus non-rigid log10 rate {c['estimate']:.3f}, p {c['p']:.2g}")
print("Expected units: HO 1048, MO 1179, MS 786, PR 1048")
print("All sorted units by type (main recordings):",
      pd.Series({k: R[k]["U"] for k in RECS}).groupby(lambda k: R[k]["mtype"]).sum().to_dict())

**Result: reproduced.** Backbone units fire faster than non-rigid units in every model type
(log10 rate differences 0.35 to 0.81, p below 1e-30), at both slice thresholds. The unit counts
differ from the legend's: 699, 624, 813 and 1744 units with at least 30 spikes (HO, MO, MS, PR)
against 1048, 1179, 786 and 1048; the four murine organoids without bursts are left out.

In [ ]:
for sk in ["bb", "bb_70"]:
    d, f = f7a[sk]
    out = {}
    for g in ["bb", "nr"]:
        for t in ["HO", "MO", "MS"]:
            c = sq.cell_means_contrast(f, f"PR_{g}", f"{t}_{g}")
            out[f"PR-{t} {g}"] = (round(c["estimate"], 3), float(f"{c['p']:.2g}"))
    print(f"Slices at {'90' if sk == 'bb' else '70'}%:", out)
    print("   favouring primaries with p < 0.05:", sum(v[0] > 0 and v[1] < 0.05 for v in out.values()), "of 6")
print("Median rate by type and unit group:", d.groupby(["mtype", "bb"]).rate.median().round(2).unstack().to_dict("index"))

**Result: partially reproduced.** Five of the six contrasts favour primary cultures with p <
0.05; primary backbone units do not fire faster than murine organoid backbone units (difference
0.04 log10 units, p = 0.66 and 0.68).

### fig7b-normalized-b2b and fig7b-primary-no-b2b-difference

In [ ]:
f7b = {}
for sk in ["bb", "bb_70"]:
    ua = unit_all(sk)
    for norm in ["ratio", "difference"]:
        d = ua[np.isfinite(ua.b2b30) & np.isfinite(ua.b2b_shuf)].copy()
        d["nb"] = sq.normalize(d.b2b30, d.b2b_shuf) if norm == "ratio" else d.b2b30 - d.b2b_shuf
        f, con = within_type(d, "nb")
        f7b[(sk, norm)] = (d, f, con)
        cross = {t: sq.cell_means_contrast(f, f"{t}_bb", "PR_bb") for t in ["HO", "MO", "MS"]}
        print(f"Slices {'90' if sk == 'bb' else '70'}%, {norm}: units {len(d)}")
        print("   backbone minus non-rigid:", {t: (round(c["estimate"], 3), float(f"{c['p']:.2g}")) for t, c in con.items()})
        print("   backbone minus primary backbone:", {t: (round(c["estimate"], 3), float(f"{c['p']:.2g}")) for t, c in cross.items()})

**Result: partially reproduced.** After normalisation, backbone units are more correlated than
non-rigid units in human organoids (p = 2e-8), murine organoids (p = 7e-5) and slices (p below
2e-9), under both normalisations and slice thresholds. Backbone units of these types do not exceed
primary-culture backbone units with the ratio normalisation (p = 0.11, 0.41, 0.22); with the
difference normalisation only human organoids do (p = 0.025).

In [ ]:
for (sk, norm), (d, f, con) in f7b.items():
    c = con["PR"]
    print(f"Slices {'90' if sk == 'bb' else '70'}%, {norm}: primary backbone minus non-rigid {c['estimate']:.3f}, p {c['p']:.2g}")

**Result: reproduced.** In primary cultures normalised burst-to-burst correlations do not differ
between backbone and non-rigid units (-0.003, p = 0.76; difference normalisation p = 0.70).

### fig7c (normalized pairwise correlations)

In [ ]:
pair_parts = []
for sk in ["bb", "bb_70"]:
    for k in ALL:
        key = sk if R[k]["mtype"] == "MS" else "bb"
        p = pair_table([k], key)
        p["slice_key"] = sk
        pair_parts.append(p)
pa = pd.concat(pair_parts, ignore_index=True)
pa = pa[np.isfinite(pa.y) & np.isfinite(pa.shuf)]
pa["norm"] = sq.normalize(pa.y, pa.shuf)
pa["diff"] = pa.y - pa.shuf
print(pa[pa.slice_key == "bb"].groupby(["mtype", "type"]).norm.agg(["count", "median"]).round(3).unstack().to_string())


def median_model(sk, v, types=("HO", "MO", "MS", "PR"), pair_level=False):
    d = pa[(pa.slice_key == sk) & pa.type.isin(["bb", "nr"])]
    if not pair_level:
        d = d.groupby(["rec", "mtype", "type"])[v].median().reset_index()
    d = d.assign(cell=d.mtype + "_" + d.type)
    f = sq.lmm(d.rename(columns={v: "yy"}), "yy ~ 0 + C(cell)", "rec")
    names = set(f.fe_params.index)
    return {t: sq.cell_means_contrast(f, f"{t}_bb", f"{t}_nr") for t in types
            if f"C(cell)[{t}_bb]" in names and f"C(cell)[{t}_nr]" in names}


f7c = {}
for sk in ["bb", "bb_70"]:
    for v in ["norm", "diff"]:
        for pl in [False, True]:
            f7c[(sk, v, pl)] = median_model(sk, v, pair_level=pl)

In [ ]:
for (sk, v, pl), con in f7c.items():
    print(f"slices {'90' if sk == 'bb' else '70'}%, {v}, {'pair level' if pl else 'per-recording medians'}:",
          {t: (round(c["estimate"], 3), float(f"{c['p']:.2g}")) for t, c in con.items()})

**Result: reproduced.** On per-recording medians of the ratio-normalised correlations,
backbone pairs exceed non-rigid pairs in human organoids (0.163, p = 2e-4) and murine organoids
(0.237, p = 7e-6, five recordings with bursts), at both slice thresholds; the difference
normalisation and the pair-level variant agree.

In [ ]:
for (sk, v, pl), con in f7c.items():
    if v == "norm" and not pl:
        print(f"slices {'90' if sk == 'bb' else '70'}%: MS {con['MS']['estimate']:.3f} p {con['MS']['p']:.2g}; "
              f"PR {con['PR']['estimate']:.3f} p {con['PR']['p']:.2g}")

**Result: reproduced.** On the same model, backbone and non-rigid pairs do not differ in slices
(0.057, p = 0.24; 0.059, p = 0.27 at 70%) or primary cultures (0.031, p = 0.46). The pair-level
variant finds both significant (p below 1e-25).

In [ ]:
for sk in ["bb", "bb_70"]:
    out = {}
    for t in ["HO", "MO", "MS", "PR"]:
        d = pa[(pa.slice_key == sk) & (pa.mtype == t) & (pa.type == "bb")]
        f = sq.lmm(d, "norm ~ 1", "rec")
        est, p2 = f.params["Intercept"], f.pvalues["Intercept"]
        p1 = p2 / 2 if est > 0 else 1 - p2 / 2
        out[t] = dict(pairs=len(d), recordings=d.rec.nunique(), mean=round(est, 4), p_one_sided=float(f"{p1:.2g}"),
                      p_two_sided=float(f"{p2:.2g}"))
    print(f"slices {'90' if sk == 'bb' else '70'}%:")
    print(pd.DataFrame(out).T.to_string())

**Result: not reproduced.** Normalised backbone pair correlations are above 0 in human
organoids (0.162, one-sided p = 6e-9) but not in murine organoids (0.008, p = 0.35), and in slices
only at the 90% threshold (0.073, p = 0.043; 0.063, p = 0.088 at 70%). In primary cultures they
are above 0 (0.043, p = 0.004), where the paper finds no effect.

### fig7d lag times

In [ ]:
lagd = pa[(pa.slice_key == "bb") & (pa.type == "bb")].copy()
print(lagd.groupby("mtype").lag.describe().round(1).to_string())
f7d = {}
for reading in ["a_drop_zero", "b_zero_as_1ms"]:
    d = lagd[lagd.lag > 0] if reading == "a_drop_zero" else lagd.assign(lag=lagd.lag.clip(lower=1))
    d = d.assign(ll=np.log10(d.lag))
    f = sq.lmm(d, "ll ~ 0 + C(mtype)", "rec")
    f7d[reading] = {f"{a}-{b}": sq.contrast(f, {f"C(mtype)[{a}]": 1.0, f"C(mtype)[{b}]": -1.0})
                    for a, b in [("HO", "PR"), ("MO", "PR"), ("HO", "MO"), ("HO", "MS"), ("MO", "MS"), ("MS", "PR")]}
    print(f"{reading}: {len(d)} backbone pairs; log10 lag means", d.groupby("mtype").ll.mean().round(3).to_dict())
    print("   ", {k: (round(c["estimate"], 3), float(f"{c['p']:.2g}")) for k, c in f7d[reading].items()})

In [ ]:
for reading, con in f7d.items():
    print(reading, {k: (round(con[k]["estimate"], 3), float(f"{con[k]['p']:.2g}")) for k in ["HO-PR", "MO-PR", "HO-MO", "HO-MS"]})

**Result: partially reproduced.** Backbone lags are longer in human organoids (0.86 log10 units,
p = 1e-11) and murine organoids (0.70, p = 4e-7) than in primary cultures, and human organoids
differ from slices (p = 7e-4), but not from murine organoids (0.15, p = 0.28). Setting zero lags
to 1 ms gives the same pattern. The code's lag table (Processed_data/Figure7/lag_data.xlsx) has
no zero lag, so reading (a), dropping them, is the code's.

In [ ]:
for reading, con in f7d.items():
    print(reading, {k: (round(con[k]["estimate"], 3), float(f"{con[k]['p']:.2g}")) for k in ["MO-MS", "MS-PR"]})

**Result: not reproduced.** Under reading (a), the code's, murine organoid lags are longer than
slice lags (0.31 log10 units, p = 0.035); with zero lags set to 1 ms, p = 0.059. Slices and
primary cultures differ (p = 0.004).

### fig7e-pca-backbone-variance

In [ ]:
rows = []
for k in ALL:
    r = R[k]
    p = r["pca"]
    if any(p[s] is None for s in ["all", "bb", "nr"]):
        continue
    A = sq.pca_ratio(p["all"], p["bb"], p["nr"])
    Bs = [sq.pca_ratio(x["all"], x["bb"], x["nr"]) for x in r["shuf"]["pca"]
          if all(x[s] is not None for s in ["all", "bb", "nr"])]
    B = np.mean(Bs, 0)
    for i, s in enumerate(["all", "bb", "nr"]):
        rows.append(dict(rec=k, mtype=r["mtype"], sel=s, value=A[i], shuf=B[i], norm=sq.normalize(A[i], B[i]), diff=A[i] - B[i]))
pc = pd.DataFrame(rows)
print(pc.pivot_table(index=["mtype", "rec"], columns="sel", values="norm").round(3).to_string())
for v in ["norm", "diff"]:
    for reading, other in [("a", "nr"), ("b", "all")]:
        d = pc[pc.sel.isin(["bb", other])] if reading == "a" else pc
        d = d.assign(cell=d.mtype + "_" + d.sel)
        f = sq.lmm(d.rename(columns={v: "yy"}), "yy ~ 0 + C(cell)", "rec")
        con = {t: sq.cell_means_contrast(f, f"{t}_bb", f"{t}_{other}") for t in ["HO", "MO", "MS", "PR"]}
        print(f"{v}, reading ({reading}) backbone minus {other}:", {t: (round(c["estimate"], 3), float(f"{c['p']:.2g}")) for t, c in con.items()})
d = pc[pc.sel == "bb"]
f = sq.lmm(d, "norm ~ 0 + C(mtype)", "rec")
print("Backbone, organoids against other types:",
      {f"{a}-{b}": float(f"{sq.contrast(f, {f'C(mtype)[{a}]': 1.0, f'C(mtype)[{b}]': -1.0})['p']:.2g}") for a in ["HO", "MO"] for b in ["MS", "PR"]})

**Result: not reproduced.** The normalised three-PC variance of backbone units exceeds that of
non-rigid units in human organoids (0.187, p = 4e-11) but not in murine organoids (-0.040, p =
0.23); against all units, human organoids again (p = 1e-8) and murine organoids not (p = 0.23),
while primary cultures are (p = 0.046). Human organoid backbone units exceed those of slices (p =
0.006) and primary cultures (p = 1e-4); murine organoid ones do not.

### supp24c-nonrigid-more-variable

In [ ]:
for sk in ["bb", "bb_70"]:
    ua = unit_all(sk)
    d = ua[(ua.n_spk >= sq.MIN_SPIKES) & np.isfinite(ua.cv2)].copy()
    d["cv2z"] = (d.cv2 - d.cv2_shuf_mean) / d.cv2_shuf_sd
    for v in ["cv2", "cv2z"]:
        dd = d[np.isfinite(d[v])]
        f, con = within_type(dd, v)
        print(f"slices {'90' if sk == 'bb' else '70'}%, {v}: non-rigid minus backbone",
              {t: (round(-c["estimate"], 3), float(f"{c['p']:.2g}")) for t, c in con.items()})

**Result: reproduced.** Non-rigid units have higher CV2 than backbone units in all four model
types, raw (p from 1e-6 to 7e-40) and z-scored to the shuffles (p from 0.02 to 7e-55), at both
slice thresholds.

### supp10b-rank-order-by-model

In [ ]:
rk_all = rank_table(R, ALL)
rk_all["mtype"] = rk_all.rec.map(lambda k: R[k]["mtype"])
print(rk_all.groupby("mtype")[["raw", "norm", "z"]].median().round(3).to_string())
for v in ["norm", "z"]:
    d = rk_all[np.isfinite(rk_all[v])]
    f = sq.lmm(d, f"{v} ~ 0 + C(mtype)", "rec")
    names = list(f.fe_params.index)
    Wm = np.array([[1, -1, 0, 0], [1, 0, -1, 0], [1, 0, 0, -1]], float)
    cov = f.cov_params().loc[names, names].values
    est = Wm @ f.fe_params.values
    chi2 = float(est @ np.linalg.solve(Wm @ cov @ Wm.T, est))
    c = sq.contrast(f, {"C(mtype)[HO]": 1.0, "C(mtype)[PR]": -1.0})
    print(f"{v}: {len(d)} burst pairs in {d.rec.nunique()} recordings; main effect chi2(3) {chi2:.1f}, p {stats.chi2.sf(chi2, 3):.2g}; "
          f"HO minus PR {c['estimate']:.3f}, p {c['p']:.2g}")

**Result: blocked (methods underspecified).** With z-scored rank-order correlations the model
type matters (chi2(3) = 10.5, p = 0.015) and human organoids exceed primary cultures (1.37, p =
0.033); with the ratio normalisation neither holds (p = 0.65 and 0.45), for the reason given
under ed4d (shuffled correlations near 0). The criterion asks for both readings.

## The two sets of human organoids (ed2d, ed2e, ed2f)

In [ ]:
for name, recs in [("Sharf (HO1-HO4)", HO4), ("Alam El Din (HO5-HO8)", ["HO5", "HO6", "HO7", "HO8"])]:
    u = unit_table(recs)
    d = u[u.n_spk >= sq.MIN_SPIKES].assign(lr=lambda x: np.log10(x.rate))
    f = sq.lmm(d, "lr ~ bb", "rec")
    print(f"{name}: {len(d)} units ({int(d.bb.sum())} backbone); backbone minus non-rigid log10 rate {f.params['bb']:.3f}, p {f.pvalues['bb']:.2g}")

**Result: not reproduced.** Backbone units fire faster in the Sharf recordings (0.72 log10
units, p = 1e-77), and also in the Alam El Din recordings (0.12, p = 0.0018), where the paper
finds no difference (P = 0.40).

In [ ]:
for name, recs in [("Sharf (HO1-HO4)", HO4), ("Alam El Din (HO5-HO8)", ["HO5", "HO6", "HO7", "HO8"])]:
    u = unit_table(recs)
    d = u[np.isfinite(u.b2b30) & np.isfinite(u.b2b_shuf)].assign(nb=lambda x: sq.normalize(x.b2b30, x.b2b_shuf))
    f = sq.lmm(d, "nb ~ bb", "rec")
    print(f"{name}: {len(d)} units; normalised b2b backbone {d[d.bb == 1].nb.mean():.3f}, non-rigid {d[d.bb == 0].nb.mean():.3f}; "
          f"effect {f.params['bb']:.3f}, p {f.pvalues['bb']:.2g}")

**Result: reproduced.** Normalised burst-to-burst correlations are higher for backbone units in
both sets (Sharf 0.040, p = 0.005; Alam El Din 0.084, p = 1e-5).

In [ ]:
for name, recs in [("Sharf (HO1-HO4)", HO4), ("Alam El Din (HO5-HO8)", ["HO5", "HO6", "HO7", "HO8"])]:
    out = {}
    for t in ["bb", "mix", "nr"]:
        d = pa[(pa.slice_key == "bb") & pa.rec.isin(recs) & (pa.type == t)]
        if d.rec.nunique() < 2:
            out[t] = dict(pairs=len(d))
            continue
        f = sq.lmm(d, "norm ~ 1", "rec")
        est, p2 = f.params["Intercept"], f.pvalues["Intercept"]
        out[t] = dict(pairs=len(d), recordings=d.rec.nunique(), mean=round(est, 4),
                      p_one_sided=float(f"{(p2 / 2 if est > 0 else 1 - p2 / 2):.2g}"))
    print(name)
    print(pd.DataFrame(out).T.to_string())

**Result: partially reproduced.** Backbone pairs are above 0 in both sets (Sharf p = 3e-6, Alam
El Din p = 8e-4), and non-rigid pairs are not; mixed pairs are also above 0 in the Sharf
recordings (0.087, p = 0.009), where the paper finds P = 0.145.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| methods-burst-count | reproduced (mean only; recordings range from 0 to 156 bursts) |
| fig1c-example-burst-count | reproduced (HO1, 46 bursts) |
| ed1c-lognormal-rates | reproduced (reading a, all units) |
| fig1d-burst-duration | reproduced |
| fig1e-peaks-near-burst-peak | partially reproduced (15 of 16 readings) |
| fig2-backbone-fraction | reproduced (units with at least 30 spikes) |
| fig2e-b2b-correlation | reproduced |
| fig2g-variance-latency | reproduced |
| ed3f-burst-cluster-variability | reproduced |
| ed4b-gabazine-more-bursts | reproduced |
| ed4b-nbqx-cpp-abolish-bursts | reproduced |
| ed4c-gabazine-participation | reproduced |
| ed4d-gabazine-rank-order | blocked: methods underspecified (the two normalisations disagree) |
| supp2a-backbone-fraction-declines | partially reproduced (human only) |
| supp2c-b2b-increases-with-age | partially reproduced (human only) |
| supp8-backbone-stable-4h | blocked: units not identified across recordings |
| ed5-shuffle-reduces-b2b | reproduced |
| fig3e-pairwise-correlation | reproduced |
| fig3f-backbone-pairs-in-tail | partially reproduced |
| fig4c-similarity-variance | reproduced |
| fig4d-burst-similarity | reproduced |
| fig4e-similarity-rises-at-backbone-onset | partially reproduced (HO7 has no backbone) |
| supp13-top-correlated-units-similarity | reproduced |
| fig4f-pca-variance | partially reproduced |
| fig6f-slice-b2b-correlation | reproduced |
| fig6g-slice-pairwise-correlation | reproduced |
| fig7a-backbone-higher-rate | reproduced |
| fig7a-primary-higher-rate | partially reproduced |
| fig7b-normalized-b2b | partially reproduced |
| fig7b-primary-no-b2b-difference | reproduced |
| fig7c-normalized-pairwise | reproduced |
| fig7c-slices-primaries-no-pair-difference | reproduced |
| fig7c-backbone-pairs-above-random | not reproduced |
| fig7d-lag-times | partially reproduced |
| fig7d-murine-organoids-slices-same-lag | not reproduced |
| fig7e-pca-backbone-variance | not reproduced |
| supp24c-nonrigid-more-variable | reproduced |
| supp10b-rank-order-by-model | blocked: methods underspecified (as ed4d) |
| ed2d-alam-el-din-rates | not reproduced |
| ed2e-alam-el-din-b2b | reproduced |
| ed2f-backbone-pairs-above-zero | partially reproduced |
| HMM claims (supp18, fig5c, fig5d, fig5f, fig7f, supp24d, supp26) | in `hmm_van_der_molen_2025.ipynb` |
| supp27-d2-near-critical | blocked (not run in this attempt) |
| supp5-murine-organoid-forebrain-identity | blocked (data elsewhere; markers not listed) |

**How the analysis differs from the paper's.** 10 shuffles per recording (3 for pairwise
correlations and shuffled similarity) where the paper uses 100; burst-only rates that do not
hold the inverse ISI across the gap between bursts; mixed models fitted by REML here with Wald z
tests, where the paper uses lmerTest; MATLAB smoothing replaced by its scipy counterparts (Gaussian
with standard deviation one fifth of the window); the shuffles cover the analysed segment (the
code shuffles the first 180 s).

**What was found about the dataset.** Every recording the paper analyses is in the dandiset as a
sorted file with a 1 ms spike matrix; the backbone units and burst counts of HO1 to HO4 equal the
numbers in the paper's processed tables (27, 26, 23 and 15 backbone units; 46, 31, 42 and 28
bursts). The HO2 and HO3 longitudinal recordings are deposited two or three times. The baseline
sorted file of MO12 and all murine organoid longitudinal recordings are missing. Units carry
positions but no identifier linking them across recordings of one organoid. Pr1 to Pr4 are
labelled rat, the paper calls them murine. With the stated burst detection, four murine organoids
have no burst, so the paper's murine organoid parameters are not given.